
# Roxy notebook example: Amino acid composition (AAC)

This notebook is a **reference implementation example** for the **AAC descriptor family** in Roxy.

It uses a small demo dataset and shows how to compute **amino acid composition descriptors** from raw protein sequences, using a clear implementation style that can later be migrated into the real package.

## Covered AAC outputs

This notebook implements:

- sequence cleaning
- validation against the 20 standard amino acids
- absolute amino acid counts
- normalized amino acid frequencies
- optional cumulative sanity checks
- export-ready AAC tables
- a simple class-style implementation that mirrors how this could look inside Roxy

The main goal is to provide a **clean teaching notebook** that the student can directly use as a starting point for implementation.


In [1]:

from collections import Counter

import numpy as np
import pandas as pd



## Demo dataset

This is just a tiny synthetic dataset for implementation examples.


In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "aac_demo_1",
            "aac_demo_2",
            "aac_demo_3",
            "aac_demo_4",
            "aac_demo_5",
            "aac_demo_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,aac_demo_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,aac_demo_2,GGGGGGGGGGGGGGG,B
2,aac_demo_3,KRRKRRKRRKRRDDDDEE,A
3,aac_demo_4,ACDEFGHIKLMNPQRSTVWY,B
4,aac_demo_5,PPPPGSSSSSTTTTNNQQQ,A
5,aac_demo_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    """Clean a protein sequence, keeping only standard amino acids."""
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq


def amino_acid_counts(seq: str) -> dict:
    """Return absolute counts for the 20 standard amino acids."""
    seq = clean_sequence(seq)
    counts = Counter(seq)
    return {f"aac_count_{aa}": counts.get(aa, 0) for aa in STANDARD_AA}


def amino_acid_frequencies(seq: str) -> dict:
    """Return normalized amino acid frequencies for the 20 standard amino acids."""
    seq = clean_sequence(seq)
    length = len(seq)
    if length == 0:
        return {f"aac_freq_{aa}": np.nan for aa in STANDARD_AA}
    counts = Counter(seq)
    return {f"aac_freq_{aa}": counts.get(aa, 0) / length for aa in STANDARD_AA}


def amino_acid_composition(seq: str, include_counts: bool = True, include_frequencies: bool = True) -> dict:
    """Compute a combined AAC descriptor dictionary."""
    seq = clean_sequence(seq)
    out = {
        "aac_length": len(seq),
        "aac_valid_residue_count": len(seq),
        "aac_unique_residue_count": len(set(seq)),
    }

    if include_counts:
        out.update(amino_acid_counts(seq))
    if include_frequencies:
        out.update(amino_acid_frequencies(seq))

    if include_frequencies and len(seq) > 0:
        out["aac_frequency_sum"] = sum(out[f"aac_freq_{aa}"] for aa in STANDARD_AA)
    else:
        out["aac_frequency_sum"] = np.nan

    return out


## Simple functional usage

In [5]:

aac_example = amino_acid_composition(df_demo.loc[0, "sequence"])
aac_example


{'aac_length': 24,
 'aac_valid_residue_count': 24,
 'aac_unique_residue_count': 13,
 'aac_count_A': 1,
 'aac_count_C': 0,
 'aac_count_D': 0,
 'aac_count_E': 0,
 'aac_count_F': 4,
 'aac_count_G': 1,
 'aac_count_H': 0,
 'aac_count_I': 1,
 'aac_count_K': 1,
 'aac_count_L': 3,
 'aac_count_M': 1,
 'aac_count_N': 0,
 'aac_count_P': 0,
 'aac_count_Q': 0,
 'aac_count_R': 3,
 'aac_count_S': 4,
 'aac_count_T': 1,
 'aac_count_V': 2,
 'aac_count_W': 1,
 'aac_count_Y': 1,
 'aac_freq_A': 0.041666666666666664,
 'aac_freq_C': 0.0,
 'aac_freq_D': 0.0,
 'aac_freq_E': 0.0,
 'aac_freq_F': 0.16666666666666666,
 'aac_freq_G': 0.041666666666666664,
 'aac_freq_H': 0.0,
 'aac_freq_I': 0.041666666666666664,
 'aac_freq_K': 0.041666666666666664,
 'aac_freq_L': 0.125,
 'aac_freq_M': 0.041666666666666664,
 'aac_freq_N': 0.0,
 'aac_freq_P': 0.0,
 'aac_freq_Q': 0.0,
 'aac_freq_R': 0.125,
 'aac_freq_S': 0.16666666666666666,
 'aac_freq_T': 0.041666666666666664,
 'aac_freq_V': 0.08333333333333333,
 'aac_freq_W': 0.04166

## Apply AAC to the full demo dataset

In [6]:

df_aac = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(amino_acid_composition).apply(pd.Series),
    ],
    axis=1,
)

df_aac


,sequence_id,sequence,label,aac_length,aac_valid_residue_count,aac_unique_residue_count,aac_count_A,aac_count_C,aac_count_D,aac_count_E,...,aac_freq_N,aac_freq_P,aac_freq_Q,aac_freq_R,aac_freq_S,aac_freq_T,aac_freq_V,aac_freq_W,aac_freq_Y,aac_frequency_sum
0,aac_demo_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,13.0,1.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.125000,0.166667,0.041667,0.083333,0.041667,0.041667,1.0
1,aac_demo_2,GGGGGGGGGGGGGGG,B,15.0,15.0,1.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,aac_demo_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,4.0,0.0,0.0,4.0,2.0,...,0.000000,0.000000,0.000000,0.444444,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
3,aac_demo_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,20.0,1.0,1.0,1.0,1.0,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,1.0
4,aac_demo_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,6.0,0.0,0.0,0.0,0.0,...,0.105263,0.210526,0.157895,0.000000,0.263158,0.210526,0.000000,0.000000,0.000000,1.0
5,aac_demo_6,MSTNPKPQRITLKDGNKVELV,B,21.0,21.0,14.0,0.0,0.0,1.0,1.0,...,0.095238,0.095238,0.047619,0.047619,0.047619,0.095238,0.095238,0.000000,0.000000,1.0


## Inspect AAC columns

In [7]:

aac_count_cols = [c for c in df_aac.columns if c.startswith("aac_count_")]
aac_freq_cols = [c for c in df_aac.columns if c.startswith("aac_freq_")]

len(aac_count_cols), len(aac_freq_cols)


(20, 20)

In [8]:

df_aac[["sequence_id", "aac_length", "aac_unique_residue_count", "aac_frequency_sum"] + aac_freq_cols[:10]]


,sequence_id,aac_length,aac_unique_residue_count,aac_frequency_sum,aac_freq_A,aac_freq_C,aac_freq_D,aac_freq_E,aac_freq_F,aac_freq_G,aac_freq_H,aac_freq_I,aac_freq_K,aac_freq_L
0,aac_demo_1,24.0,13.0,1.0,0.041667,0.00,0.000000,0.000000,0.166667,0.041667,0.00,0.041667,0.041667,0.125000
1,aac_demo_2,15.0,1.0,1.0,0.000000,0.00,0.000000,0.000000,0.000000,1.000000,0.00,0.000000,0.000000,0.000000
2,aac_demo_3,18.0,4.0,1.0,0.000000,0.00,0.222222,0.111111,0.000000,0.000000,0.00,0.000000,0.222222,0.000000
3,aac_demo_4,20.0,20.0,1.0,0.050000,0.05,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,0.050000
4,aac_demo_5,19.0,6.0,1.0,0.000000,0.00,0.000000,0.000000,0.000000,0.052632,0.00,0.000000,0.000000,0.000000
5,aac_demo_6,21.0,14.0,1.0,0.000000,0.00,0.047619,0.047619,0.000000,0.047619,0.00,0.047619,0.142857,0.095238


## Dataset-level AAC summary

In [9]:

aac_dataset_summary = (
    df_aac[aac_freq_cols]
    .mean(axis=0)
    .sort_values(ascending=False)
    .rename("mean_frequency")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

aac_dataset_summary.head(10)


,descriptor,mean_frequency
0,aac_freq_G,0.198653
1,aac_freq_R,0.111177
2,aac_freq_S,0.087907
3,aac_freq_K,0.076124
4,aac_freq_T,0.066239
5,aac_freq_P,0.059294
6,aac_freq_D,0.053307
7,aac_freq_L,0.045040
8,aac_freq_Q,0.042586
9,aac_freq_N,0.041750


## Sanity checks

In [10]:

assert len(aac_count_cols) == 20
assert len(aac_freq_cols) == 20
assert "aac_length" in df_aac.columns
assert "aac_frequency_sum" in df_aac.columns

valid_rows = df_aac["aac_length"] > 0
assert np.allclose(df_aac.loc[valid_rows, "aac_frequency_sum"], 1.0)

print(f"Number of AAC count descriptors: {len(aac_count_cols)}")
print(f"Number of AAC frequency descriptors: {len(aac_freq_cols)}")
print("AAC implementation sanity checks passed.")


Number of AAC count descriptors: 20
Number of AAC frequency descriptors: 20
AAC implementation sanity checks passed.


## A class-style implementation closer to the real package

In [11]:

class AminoAcidCompositionDescriptors:
    """Example class-style AAC implementation for later migration into Roxy."""

    def __init__(self, include_counts: bool = True, include_frequencies: bool = True):
        self.include_counts = include_counts
        self.include_frequencies = include_frequencies

    def transform_sequence(self, seq: str) -> dict:
        return amino_acid_composition(
            seq,
            include_counts=self.include_counts,
            include_frequencies=self.include_frequencies,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


aac_transformer = AminoAcidCompositionDescriptors(include_counts=True, include_frequencies=True)
aac_matrix = aac_transformer.transform(df_demo["sequence"].tolist())
aac_matrix.head()


,aac_length,aac_valid_residue_count,aac_unique_residue_count,aac_count_A,aac_count_C,aac_count_D,aac_count_E,aac_count_F,aac_count_G,aac_count_H,...,aac_freq_N,aac_freq_P,aac_freq_Q,aac_freq_R,aac_freq_S,aac_freq_T,aac_freq_V,aac_freq_W,aac_freq_Y,aac_frequency_sum
0,24,24,13,1,0,0,0,4,1,0,...,0.000000,0.000000,0.000000,0.125000,0.166667,0.041667,0.083333,0.041667,0.041667,1.0
1,15,15,1,0,0,0,0,0,15,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,18,18,4,0,0,4,2,0,0,0,...,0.000000,0.000000,0.000000,0.444444,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
3,20,20,20,1,1,1,1,1,1,1,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,1.0
4,19,19,6,0,0,0,0,0,1,0,...,0.105263,0.210526,0.157895,0.000000,0.263158,0.210526,0.000000,0.000000,0.000000,1.0


## Merge transformer output back to the dataset

In [12]:

df_aac_class = pd.concat([df_demo, aac_matrix], axis=1)
df_aac_class.head()


,sequence_id,sequence,label,aac_length,aac_valid_residue_count,aac_unique_residue_count,aac_count_A,aac_count_C,aac_count_D,aac_count_E,...,aac_freq_N,aac_freq_P,aac_freq_Q,aac_freq_R,aac_freq_S,aac_freq_T,aac_freq_V,aac_freq_W,aac_freq_Y,aac_frequency_sum
0,aac_demo_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,13,1,0,0,0,...,0.000000,0.000000,0.000000,0.125000,0.166667,0.041667,0.083333,0.041667,0.041667,1.0
1,aac_demo_2,GGGGGGGGGGGGGGG,B,15,15,1,0,0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,aac_demo_3,KRRKRRKRRKRRDDDDEE,A,18,18,4,0,0,4,2,...,0.000000,0.000000,0.000000,0.444444,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
3,aac_demo_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,20,1,1,1,1,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,1.0
4,aac_demo_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,6,0,0,0,0,...,0.105263,0.210526,0.157895,0.000000,0.263158,0.210526,0.000000,0.000000,0.000000,1.0



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move sequence cleaning into `roxy/sequence/cleaning.py`
- move AAC helper logic into `roxy/sequence/composition.py`
- keep the transformer class as `AminoAcidCompositionDescriptors`
- support three output modes:
  - counts only
  - frequencies only
  - counts + frequencies
- add tests for:
  - empty sequence
  - invalid characters
  - lower-case sequences
  - repeated amino-acid sequences
  - mixed composition sequences



## Optional export

Uncomment the next cell if you want to save the AAC descriptor table.


In [ ]:
# df_aac.to_csv("demo_aac_descriptors.csv", index=False)
